# Seed-threshold 0.1 WSI cleanup comparison

This is a fresh, output-isolated experiment on the existing SLIDE-0330 all-channel half crop. It uses the M11 v4 global-normalized WSI + watershed settings and changes only `seed_threshold` from 0.6 to 0.1. The first run launches the GPU pass with `RUN_WSI=True`; later runs safely reuse only a compatible completed output. Run this notebook with the `instanseg_nimbus` kernel.

The provisional cleanup is the same analysis policy used in the seed-0.6 audit: global 8-connectivity, strict nuclear component area `>10`, rejection of coordinated IDs with no surviving nucleus, removal of same-ID nucleus-free disconnected cell components, and rejection of all unnucleated cells. Every artifact is written under `connectedness_audit/seed_threshold_0p1`; the seed-0.6 Zarr is never overwritten.

The last stage renders the exact prior native hotspot window (`y=24748:25172, x=13818:14243`) and reports the removed IDs/categories there.

In [ ]:
from pathlib import Path
import json, inspect, re, subprocess, sys, time
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from skimage.segmentation import find_boundaries

MIF_PIPELINE_ROOT = Path('/data1/lowes/ratnayn/Codex/projects/mIF-pipeline')
INSTANSEG_ROOT = Path('/data1/lowes/ratnayn/Codex/projects/instanseg')
CROP = Path('/data1/lowes/ratnayn/Codex/codex-scratch/mIF-pipeline/instanseg_watershed_production_smoke_all_channel_crop/SLIDE-0330/SLIDE-0330_all_channels_half_crop.ome.tif')
BASE_AUDIT_DIR = CROP.parent / 'connectedness_audit'
OUTPUT_DIR = BASE_AUDIT_DIR / 'seed_threshold_0p1'
RESOLVED_ZARR = OUTPUT_DIR / 'SLIDE-0330_watershed_resolved_seed_threshold_0p1.zarr'
CLEANED_ZARR = OUTPUT_DIR / 'SLIDE-0330_watershed_resolved_seed_threshold_0p1_postresolution_8conn_min10.zarr'
PER_ID_CSV = OUTPUT_DIR / 'SLIDE-0330_seed_threshold_0p1_cleanup_per_id.csv'
SUMMARY_JSON = OUTPUT_DIR / 'SLIDE-0330_seed_threshold_0p1_cleanup_summary.json'
REMOVAL_OVERVIEW_PNG = OUTPUT_DIR / 'SLIDE-0330_seed_threshold_0p1_removed_overview.png'
FIXED_HOTSPOT_PNG = OUTPUT_DIR / 'SLIDE-0330_seed_threshold_0p1_fixed_hotspot_preview.png'
FIXED_WINDOW_JSON = OUTPUT_DIR / 'SLIDE-0330_seed_threshold_0p1_fixed_hotspot_metrics.json'

# Fresh first run; a compatible completed output is reused on later runs.
RUN_WSI = True
REUSE_COMPLETED_WSI = True
RUN_CLEANUP = True
CLEANUP_REUSE = True
CLEANUP_OVERWRITE = True
AUDIT_CHUNK = 2048

MODEL_NAME = 'fluorescence_nuclei_and_cells'
PIXEL_SIZE_UM = 0.325
NORMALIZATION_PERCENTILES = (0.1, 99.9)
WSI_TILE_SIZE, WSI_OVERLAP, WSI_DETECTION_SIZE, WSI_BATCH_SIZE = 2048, 80, 20, 1
SEED_THRESHOLD = 0.1
REFERENCE_CHANNEL = 'R1_DAPI'
# Exact ten-alias order used by the M11 v4 WSI configuration.
SEGMENTATION_CHANNELS = [
    'R1_DAPI', 'R4_P19_POLYRAT', 'R4_GFP_POLY_AF488',
    'R6_CD45_CST_AF647', 'R6_PANCK_AE1_AE3_750',
    'R12_CD31_D8V9E_AF750', 'R7_NAK_ATPASE_555',
    'R8_F480_D2S9R_555', 'R9_CD68_E3O7V_488',
    'R12_CD3E_E4T1B_AF555',
]
# The previous preview was based on this exact native window.
FIXED_NATIVE_BOUNDS = (24748, 25172, 13818, 14243)  # y0, y1, x0, x1
PRIOR_MODEL_BOUNDS = (16128, 16320, 9024, 9216)

assert OUTPUT_DIR.resolve() != BASE_AUDIT_DIR.resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if str(INSTANSEG_ROOT) not in sys.path:
    sys.path.insert(0, str(INSTANSEG_ROOT))
print({'crop': str(CROP), 'output_dir': str(OUTPUT_DIR), 'run_wsi': RUN_WSI, 'seed_threshold': SEED_THRESHOLD})

## 1. Preflight provenance and alias resolution

This cell verifies the editable InstanSeg fork, applies the required TiffSlide patch, and resolves aliases from the input OME metadata. It does not run inference.

In [ ]:
if not CROP.is_file():
    raise FileNotFoundError(CROP)
import zarr
import instanseg
from instanseg import InstanSeg
from tiffslide import TiffSlide
import instanseg.inference_class as inference_class
inference_class.TiffSlide = TiffSlide
fork_path = Path(instanseg.__file__).resolve()
fork_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=INSTANSEG_ROOT, text=True).strip()
fork_status = subprocess.check_output(['git', 'status', '--porcelain'], cwd=INSTANSEG_ROOT, text=True).strip()
assert fork_path.is_relative_to(INSTANSEG_ROOT), fork_path
assert hasattr(InstanSeg, 'eval_whole_slide_image_global_normalization')
method_signature = inspect.signature(InstanSeg.eval_whole_slide_image_global_normalization)
method_parameters = set(method_signature.parameters)
required_parameters = {'channel_ids', 'pixel_size', 'normalization_percentiles', 'reference_channel_id', 'tile_size', 'overlap', 'detection_size', 'batch_size', 'output_path', 'resolve_cell_and_nucleus', 'resolution_method', 'allow_unnucleated_cells'}
assert required_parameters.issubset(method_parameters), sorted(required_parameters - method_parameters)
assert any(parameter.kind is inspect.Parameter.VAR_KEYWORD for parameter in method_signature.parameters.values()), 'cleanup_fragments and seed_threshold must be accepted through **kwargs'

with tifffile.TiffFile(CROP) as tif:
    series = tif.series[0]
    crop_shape, crop_axes = tuple(int(v) for v in series.shape), series.axes
    ome_xml = tif.ome_metadata or ''
channel_names = [m.group(1) for m in re.finditer(r'<(?:[^:>]+:)?Channel\b[^>]*?Name="([^"]*)"', ome_xml)]
if crop_axes != 'CYX' or len(channel_names) != crop_shape[0]:
    raise ValueError(f'Expected named CYX input, got axes={crop_axes}, shape={crop_shape}, names={len(channel_names)}')
if len(channel_names) != len(set(channel_names)):
    raise ValueError('OME channel names are not unique.')
lookup = {name: index for index, name in enumerate(channel_names)}
missing = [name for name in SEGMENTATION_CHANNELS if name not in lookup]
if missing:
    raise KeyError(f'Missing v4 segmentation aliases: {missing}')
CHANNEL_IDS = [lookup[name] for name in SEGMENTATION_CHANNELS]
REFERENCE_CHANNEL_ID = lookup[REFERENCE_CHANNEL]
assert CHANNEL_IDS[0] == REFERENCE_CHANNEL_ID
print({'python': sys.executable, 'instanseg': str(fork_path), 'commit': fork_commit, 'dirty': bool(fork_status), 'crop_shape': crop_shape, 'channels': list(zip(SEGMENTATION_CHANNELS, CHANNEL_IDS)), 'reference_channel_id': REFERENCE_CHANNEL_ID})

## 2. Fresh seed-threshold 0.1 WSI + watershed pass

Completed output is reused only when its provenance matches this crop, channel order, resolver, WSI geometry, normalization, and seed threshold. Existing incompatible output is never overwritten.

In [ ]:
def _seed_setting(attrs):
    for obj in (attrs.get('wsi_settings') or {}, attrs.get('resolution') or {}, attrs.get('eval_kwargs') or {}, attrs.get('inference') or {}):
        if 'seed_threshold' in obj:
            return float(obj['seed_threshold'])
    return None

def compatible_completed_output(path):
    if not path.is_dir():
        return False
    try:
        arr = zarr.open(str(path), mode='r')
        attrs = dict(arr.attrs)
        settings = attrs.get('wsi_settings') or {}
        normalization = attrs.get('normalization') or {}
        resolution = attrs.get('resolution') or {}
        return (attrs.get('status') == 'complete' and list(attrs.get('planes', [])) == ['nuclei', 'cells']
                and Path(attrs.get('source_image', '')).resolve() == CROP.resolve()
                and list(attrs.get('channel_ids', [])) == CHANNEL_IDS
                and settings.get('tile_size') == WSI_TILE_SIZE and settings.get('overlap') == WSI_OVERLAP
                and settings.get('detection_size') == WSI_DETECTION_SIZE
                and settings.get('resolve_cell_and_nucleus') is True
                and settings.get('resolution_method') == 'watershed'
                and [float(v) for v in normalization.get('percentiles', [])] == list(NORMALIZATION_PERCENTILES)
                and resolution.get('method') == 'watershed'
                and resolution.get('allow_unnucleated_cells') is True
                and settings.get('seed_threshold') == SEED_THRESHOLD
                and resolution.get('seed_threshold') == SEED_THRESHOLD
                and attrs.get('instanseg_fork') == str(INSTANSEG_ROOT.resolve())
                and attrs.get('instanseg_fork_commit') == fork_commit
                and attrs.get('experiment_provenance', {}).get('seed_threshold') == SEED_THRESHOLD)
    except Exception:
        return False

if compatible_completed_output(RESOLVED_ZARR) and REUSE_COMPLETED_WSI:
    print('Reusing compatible seed-0.1 output:', RESOLVED_ZARR)
elif RUN_WSI:
    if RESOLVED_ZARR.exists():
        raise ValueError(f'Existing output is incompatible; choose a new target or inspect it first: {RESOLVED_ZARR}')
    model = InstanSeg(MODEL_NAME, verbosity=1)
    started = time.perf_counter()
    observed = model.eval_whole_slide_image_global_normalization(
        str(CROP), channel_ids=CHANNEL_IDS, pixel_size=PIXEL_SIZE_UM,
        normalization_percentiles=NORMALIZATION_PERCENTILES, reference_channel_id=REFERENCE_CHANNEL_ID,
        tile_size=WSI_TILE_SIZE, overlap=WSI_OVERLAP, detection_size=WSI_DETECTION_SIZE,
        batch_size=WSI_BATCH_SIZE, output_path=RESOLVED_ZARR, overwrite=False,
        resolve_cell_and_nucleus=True, resolution_method='watershed',
        allow_unnucleated_cells=True, cleanup_fragments=True, seed_threshold=SEED_THRESHOLD,
    )
    assert Path(observed).resolve() == RESOLVED_ZARR.resolve()
    # The current fork does not persist every eval kwarg in WSI attrs; stamp
    # the exact experiment settings so later reuse cannot silently accept a
    # seed-0.6 (or another-fork) artifact at this path.
    stamped = zarr.open(str(RESOLVED_ZARR), mode='r+')
    stamped_wsi_settings = dict(stamped.attrs.get('wsi_settings') or {})
    stamped_wsi_settings['seed_threshold'] = SEED_THRESHOLD
    stamped.attrs['wsi_settings'] = stamped_wsi_settings
    stamped_resolution = dict(stamped.attrs.get('resolution') or {})
    stamped_resolution['seed_threshold'] = SEED_THRESHOLD
    stamped.attrs['resolution'] = stamped_resolution
    stamped.attrs['instanseg_fork'] = str(INSTANSEG_ROOT.resolve())
    stamped.attrs['instanseg_fork_commit'] = fork_commit
    stamped.attrs['experiment_provenance'] = {'seed_threshold': SEED_THRESHOLD, 'model': MODEL_NAME, 'pixel_size_um': PIXEL_SIZE_UM, 'channels': SEGMENTATION_CHANNELS, 'reference_channel': REFERENCE_CHANNEL, 'normalization_percentiles': list(NORMALIZATION_PERCENTILES), 'tile_size': WSI_TILE_SIZE, 'overlap': WSI_OVERLAP, 'detection_size': WSI_DETECTION_SIZE, 'batch_size': WSI_BATCH_SIZE, 'resolution_method': 'watershed', 'allow_unnucleated_cells': True, 'cleanup_fragments': True, 'instanseg_fork': str(INSTANSEG_ROOT.resolve()), 'instanseg_fork_commit': fork_commit}
    print(f'WSI seed-0.1 inference and watershed completed in {(time.perf_counter()-started)/60:.1f} min')
else:
    raise RuntimeError('No compatible seed-0.1 result exists; set RUN_WSI=True for the fresh GPU pass.')
assert compatible_completed_output(RESOLVED_ZARR)
resolved = zarr.open(str(RESOLVED_ZARR), mode='r')
print({'resolved_shape': tuple(resolved.shape), 'resolved_attrs_seed_threshold': _seed_setting(dict(resolved.attrs))})

## 3. Apply the provisional cleanup and summarize filtering

This writes a new cleaned Zarr and analysis artifacts. The inference Zarr remains read-only.

In [ ]:
if not RUN_CLEANUP:
    raise RuntimeError('Set RUN_CLEANUP=True to apply or reuse the cleanup outputs.')
helper_dir = MIF_PIPELINE_ROOT / 'notebooks'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))
from instanseg_connectedness_cleanup import run_cleanup, INSTANSEG_MIN_SIZE
cleanup_summary = run_cleanup(
    RESOLVED_ZARR, CLEANED_ZARR, PER_ID_CSV, SUMMARY_JSON, REMOVAL_OVERVIEW_PNG, CROP,
    reference_channel_id=REFERENCE_CHANNEL_ID, native_shape=crop_shape[-2:], chunk_size=AUDIT_CHUNK,
    min_size=INSTANSEG_MIN_SIZE, reuse=CLEANUP_REUSE, overwrite=CLEANUP_OVERWRITE,
)
display(pd.DataFrame([cleanup_summary['original'], cleanup_summary['final']]))
metric_keys = ('removed_nuclear_components', 'removed_nuclear_pixels', 'rejected_coordinated_ids',
               'rejected_coordinated_nuclear_pixels', 'rejected_coordinated_cell_pixels',
               'removed_nucleus_free_cell_components', 'removed_nucleus_free_cell_pixels',
               'rejected_unnucleated_ids', 'rejected_unnucleated_cell_pixels', 'removed_cell_pixels_total')
display(pd.DataFrame([{key: cleanup_summary.get(key) for key in metric_keys}]))
print({key: cleanup_summary.get(key) for key in ('cleaned_zarr', 'metrics_csv', 'metrics_json', 'overview_png')})

## 4. Exact prior native hotspot QC

The coordinates below are fixed, not redetected from the seed-0.1 removal density. Labels are mapped with the same global pixel-center formula used by the cleanup helper. The optional second row directly compares seed-0.6 artifacts when they exist.

In [ ]:
def native_label_view(label_zarr, native_bounds, native_shape):
    ny0, ny1, nx0, nx1 = map(int, native_bounds)
    native_h, native_w = map(int, native_shape[-2:])
    model_h, model_w = map(int, label_zarr.shape[-2:])
    native_y = np.arange(ny0, ny1, dtype=np.int64)
    native_x = np.arange(nx0, nx1, dtype=np.int64)
    # Global pixel-center nearest-neighbor mapping, identical to make_removed_cell_hotspot_preview.
    model_y = np.clip(((2 * native_y + 1) * model_h) // (2 * native_h), 0, model_h - 1)
    model_x = np.clip(((2 * native_x + 1) * model_w) // (2 * native_w), 0, model_w - 1)
    sy0, sx0 = int(model_y.min()), int(model_x.min())
    sy1, sx1 = int(model_y.max()) + 1, int(model_x.max()) + 1
    block = np.asarray(label_zarr[:, sy0:sy1, sx0:sx1])
    return np.take(np.take(block, model_y - sy0, axis=1), model_x - sx0, axis=2), (sy0, sy1, sx0, sx1)

def read_dapi(native_bounds):
    ny0, ny1, nx0, nx1 = map(int, native_bounds)
    with tifffile.TiffFile(str(CROP)) as tif:
        store = tif.series[0].aszarr(level=0)
        source = zarr.open(store, mode='r')
        try:
            dapi = np.asarray(source.oindex[REFERENCE_CHANNEL_ID, slice(ny0, ny1), slice(nx0, nx1)], dtype=np.float32)
        finally:
            store.close()
    low, high = np.percentile(dapi, (1.0, 99.8))
    return np.clip((dapi - low) / max(float(high - low), 1e-6), 0, 1)

native_bounds = FIXED_NATIVE_BOUNDS
dapi = read_dapi(native_bounds)
original_view, mapped_model_bounds = native_label_view(resolved, native_bounds, crop_shape[-2:])
cleaned = zarr.open(str(CLEANED_ZARR), mode='r')
cleaned_view, _ = native_label_view(cleaned, native_bounds, crop_shape[-2:])
removed_nuclei = (original_view[0] > 0) & (cleaned_view[0] == 0)
removed_cells = (original_view[1] > 0) & (cleaned_view[1] == 0)
removed_ids = np.unique(np.where(removed_cells, original_view[1], 0))
removed_ids = removed_ids[removed_ids > 0].astype(np.int64)
per_id = pd.read_csv(PER_ID_CSV)
fixed_removed = per_id.loc[per_id['label_id'].isin(removed_ids)].copy()
category_counts = fixed_removed['category'].value_counts().rename_axis('category').reset_index(name='ids')
window_summary = {'native_bounds': list(map(int, native_bounds)), 'prior_model_bounds': list(map(int, PRIOR_MODEL_BOUNDS)),
                  'mapped_model_bounds': list(map(int, mapped_model_bounds)), 'removed_cell_ids': int(len(removed_ids)),
                  'removed_nuclear_pixels': int(removed_nuclei.sum()), 'removed_cell_pixels': int(removed_cells.sum()),
                  'removed_categories': {str(k): int(v) for k, v in fixed_removed['category'].value_counts().items()},
                  'removed_ids': [int(v) for v in removed_ids]}
FIXED_WINDOW_JSON.write_text(json.dumps(window_summary, indent=2, sort_keys=True) + '\n')
display(pd.DataFrame([window_summary]).drop(columns=['removed_ids']))
display(category_counts)
display(fixed_removed[['label_id', 'category', 'original_cell_pixels', 'original_nuclear_pixels', 'removed_cell_pixels', 'removed_nuclear_pixels']].sort_values('label_id'))

cell_overlay = np.zeros(removed_cells.shape + (4,), dtype=np.uint8)
cell_overlay[removed_cells] = (0, 80, 255, 125)
nuc_overlay = np.zeros(removed_nuclei.shape + (4,), dtype=np.uint8)
nuc_overlay[removed_nuclei] = (255, 0, 0, 230)
fig, axes = plt.subplots(1, 4, figsize=(20, 5), squeeze=False)
axes = axes[0]
for ax in axes:
    ax.imshow(dapi, cmap='gray', interpolation='nearest')
    ax.set_axis_off()
axes[0].set_title('DAPI | fixed native window')
axes[1].contour(find_boundaries(original_view[1], mode='outer'), [0.5], colors=['yellow'], linewidths=0.45)
axes[1].contour(find_boundaries(original_view[0], mode='outer'), [0.5], colors=['cyan'], linewidths=0.55)
axes[1].set_title('Seed 0.1 original\nnuclei cyan; cells yellow')
axes[2].imshow(cell_overlay, interpolation='nearest')
axes[2].imshow(nuc_overlay, interpolation='nearest')
axes[2].set_title(f'Seed 0.1 removed\n{len(removed_ids)} IDs; {int(removed_cells.sum())} cell px')
axes[3].contour(find_boundaries(cleaned_view[1], mode='outer'), [0.5], colors=['yellow'], linewidths=0.45)
axes[3].contour(find_boundaries(cleaned_view[0], mode='outer'), [0.5], colors=['cyan'], linewidths=0.55)
axes[3].set_title('Seed 0.1 after cleanup')
fig.suptitle(f'Fixed hotspot native y={native_bounds[0]}:{native_bounds[1]}, x={native_bounds[2]}:{native_bounds[3]} | mapped model {mapped_model_bounds}')
fig.tight_layout()
fig.savefig(FIXED_HOTSPOT_PNG, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)
print({'preview_png': str(FIXED_HOTSPOT_PNG), 'metrics_json': str(FIXED_WINDOW_JSON)})

In [ ]:
# Optional direct two-row comparison with the previous seed-0.6 artifacts.
SEED06_RESOLVED = BASE_AUDIT_DIR / 'SLIDE-0330_watershed_resolved.zarr'
SEED06_CLEANED = BASE_AUDIT_DIR / 'SLIDE-0330_watershed_resolved_postresolution_8conn_min10.zarr'
if SEED06_RESOLVED.is_dir() and SEED06_CLEANED.is_dir():
    seed06_original = zarr.open(str(SEED06_RESOLVED), mode='r')
    seed06_cleaned = zarr.open(str(SEED06_CLEANED), mode='r')
    fig, axes = plt.subplots(2, 3, figsize=(15, 9), squeeze=False)
    for row, (tag, raw, final) in enumerate((('seed 0.1', resolved, cleaned), ('seed 0.6', seed06_original, seed06_cleaned))):
        raw_view, _ = native_label_view(raw, native_bounds, crop_shape[-2:])
        final_view, _ = native_label_view(final, native_bounds, crop_shape[-2:])
        removed_view = (raw_view[1] > 0) & (final_view[1] == 0)
        for ax in axes[row]:
            ax.imshow(dapi, cmap='gray', interpolation='nearest')
            ax.set_axis_off()
        axes[row, 0].contour(find_boundaries(raw_view[1], mode='outer'), [0.5], colors=['yellow'], linewidths=0.45)
        axes[row, 0].contour(find_boundaries(raw_view[0], mode='outer'), [0.5], colors=['cyan'], linewidths=0.55)
        axes[row, 0].set_title(f'{tag} original')
        removed_overlay = np.zeros(removed_view.shape + (4,), dtype=np.uint8)
        removed_overlay[removed_view] = (0, 80, 255, 150)
        axes[row, 1].imshow(removed_overlay, interpolation='nearest')
        axes[row, 1].set_title(f'{tag} removed ({int(removed_view.sum())} px)')
        axes[row, 2].contour(find_boundaries(final_view[1], mode='outer'), [0.5], colors=['yellow'], linewidths=0.45)
        axes[row, 2].contour(find_boundaries(final_view[0], mode='outer'), [0.5], colors=['cyan'], linewidths=0.55)
        axes[row, 2].set_title(f'{tag} cleaned')
    fig.suptitle('Same fixed native hotspot: direct seed-threshold comparison')
    fig.tight_layout()
    comparison_png = OUTPUT_DIR / 'SLIDE-0330_seed_threshold_01_vs_06_fixed_hotspot.png'
    fig.savefig(comparison_png, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print({'comparison_png': str(comparison_png)})
else:
    print('Optional seed-0.6 resolved/cleaned Zarrs were not both found; seed-0.1 QC above is still complete.')